# Imports

In [1]:
import os
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

# Read SPSS files

In [2]:
def safe_read_spss(filepath):
    try:
        df = pd.read_spss(filepath)
        df.rename(columns=str.upper, inplace=True)
        print('.', end='')
        return df
    except Exception:
        df = pd.read_spss(filepath, encoding='cp1252')
        df.rename(columns=str.upper, inplace=True)
        print('.', end='')
        return df

In [3]:
filepaths = [f for f in list(os.walk('nilt_data'))[0][2] if f.startswith('nilt')]
filepaths

['nilt20w1.sav',
 'nilt22w1.sav',
 'nilt24w1.sav',
 'nilt23w1.sav',
 'nilt21w1.sav',
 'nilt25w1.sav']

In [4]:
years = [int(f'20{fp[4:6]}') for fp in filepaths]
years

[2020, 2022, 2024, 2023, 2021, 2025]

In [5]:
df = pd.concat([safe_read_spss(f'nilt_data/{fp}') for fp in filepaths], keys=years)

......

In [6]:
df.shape

(7737, 844)

In [7]:
df = df.reset_index(0).rename(columns={'level_0': 'year'}).reset_index(drop=True)
num_years_asked = df.groupby('year').any().sum()

# Filter out questions that were only asked in one year

In [8]:
num_years_asked.value_counts()

1    552
6    150
2     80
3     22
5     22
4     18
Name: count, dtype: int64

In [9]:
multi_year_qs = num_years_asked[num_years_asked > 1]

In [10]:
df = df[['year'] + list(multi_year_qs.index)]

In [11]:
df.shape

(7737, 293)

# Data cleaning

## Standardise common duplicates

In [12]:
uniq_tot = df.stack().unique()

In [13]:
len(uniq_tot)

2500

In [14]:
lower_map = {s: s.lower() for s in uniq_tot if type(s) is str}

In [15]:
cat_cols = df.select_dtypes('category').columns

for c in cat_cols:
    df[c] = df[c].astype(str)

In [16]:
%time df.replace(lower_map, inplace=True);

CPU times: user 4.01 s, sys: 12.7 ms, total: 4.02 s
Wall time: 4.02 s


In [17]:
dont_know_map = {
    "don't know": "don't know",
    "(don't know)": "don't know",
    "i don't know": "don't know",
    'don’t know': "don't know",
    '(dont know)': "don't know",
    'dont know': "don't know",
    'i don’t know': "don't know",
    '(don’t know)': "don't know",
    "don' t know": "don't know",
    'don t know': "don't know",
    "i don' t know": "don't know",
}

In [18]:
%time df.replace(dont_know_map, inplace=True);

CPU times: user 141 ms, sys: 1.4 ms, total: 143 ms
Wall time: 142 ms


In [19]:
mixed_map = {
    'mixed-religion workplace': 'mixed religion workplace',
    'mixed-religion neighbourhood': 'mixed religion neighbourhood',
    'mixed-religion school': 'mixed religion school',
    'a mixed religion workplace': 'mixed religion workplace',
    'a mixed religion neighbourhood': 'mixed religion neighbourhood',
    'a mixed religion school': 'mixed religion school',
}

In [20]:
%time df.replace(mixed_map, inplace=True);

CPU times: user 64.2 ms, sys: 720 μs, total: 64.9 ms
Wall time: 64.4 ms


## Individual column transforms

### IRBRIT

In [21]:
nat_map = {
    'british not irish': 'British not Irish',
    'irish not british': 'Irish not British',
    'more british than irish': 'More British than Irish',
    'equally irish and british': 'Equally Irish and British',
    'more irish than british': 'More Irish than British',
    'other description (write in)': 'Other',
    "don't know": "Don't know",
    'other description (please write in)': 'Other',
}

In [22]:
df['IRBRIT'] = df['IRBRIT'].replace(nat_map)
df['IRBRIT'].value_counts(dropna=False)

IRBRIT
Irish not British            1895
British not Irish            1594
Equally Irish and British    1362
More British than Irish      1309
More Irish than British       877
Other                         484
Don't know                    147
NaN                            69
Name: count, dtype: int64

### UNINATID

In [23]:
const_map = {
    'neither': 'Neither',
    'unionist': 'Unionist',
    'nationalist': 'Nationalist',
    "don't know": "Don't know",
    'other write in': 'Other',
    'other (please write in)': 'Other',
    'other (please type in)': 'Other',
}

In [24]:
df['UNINATID'] = df['UNINATID'].replace(const_map)
df['UNINATID'].value_counts(dropna=False)

UNINATID
Neither        2966
Unionist       2481
Nationalist    1885
Don't know      238
NaN              89
Other            78
Name: count, dtype: int64

### POLPART2

In [25]:
party_map = {
    'none of these': 'None of these',
    'alliance party': 'Alliance',
    'social democratic and labour party (sdlp)': 'SDLP',
    'ulster unionist party (uup)': 'UUP',
    'democratic unionist party (dup)': 'DUP',
    "don't know": "Don't know",
    'sinn féin': 'Sinn Féin',
    'sinn fein': 'Sinn Féin',
    'dup/democratic unionist party': 'DUP',
    'other party (write in)': 'Other party',
    'other answer (write in)': 'Other party',
    'green party': 'Green party',
}

In [26]:
df['POLPART2'] = df['POLPART2'].replace(party_map)
df['POLPART2'].value_counts(dropna=False)

POLPART2
Alliance         1814
Sinn Féin        1064
DUP              1029
UUP               863
SDLP              802
None of these     673
Don't know        569
Other party       397
Green party       333
NaN               193
Name: count, dtype: int64

### RELIGCAT

In [27]:
df.replace({'protestant/other christian': 'protestant'}, inplace=True)
df['RELIGCAT'].value_counts(dropna=False)

RELIGCAT
protestant     3020
catholic       2369
no religion    2017
NaN             331
Name: count, dtype: int64

# Feature engineering

In [28]:
df['SUPPORT_ASYLUM_SEEKERS'] = df['ASYLUMNI'].dropna().apply(lambda x: 'yes' if x in ['agree', 'strongly agree'] else 'no')

In [29]:
df['SUPPORT_REFUGEES'] = df['DREFGEEB'].dropna().apply(lambda x: 'yes' if x in ['agree', 'strongly agree'] else 'no')

In [30]:
df['BELIEVE_REFUGEES_CONTRIB'] = df['RCONTRIB'].dropna().apply(lambda x: 'yes' if x in ['agree', 'strongly agree'] else 'no')

In [31]:
df['PREFER_MIXED_SCHOOL'] = df['OWNMXSCH'].dropna().apply(lambda x: 'yes' if x == 'mixed religion school' else 'no')

In [32]:
df['PREFER_MIXED_NEIGHBOURHOOD'] = df['MXRLGNGH'].apply(lambda x: 'yes' if x == 'mixed religion neighbourhood' else 'no')

In [33]:
df['PREFER_MIXED_WORKPLACE'] = df['MXRLGWRK'].apply(lambda x: 'yes' if x == 'mixed religion workplace' else 'no')

In [34]:
df['POLITICAL_PARTY'] = df['POLPART2'].apply(lambda x: x if x in ['Alliance', 'Sinn Féin', 'DUP', 'UUP', 'SDLP'] else 'Other or none')

In [35]:
df['HAS_DEGREE'] = df['HIGHQUAL2'].dropna() == 'degree level or higher'

In [36]:
df['SELF_REPORTED_LOW_INCOME'] = df['SRINC'].dropna() == 'low income'

In [37]:
df['UNDER_45'] = df['RAGECAT'].dropna().isin(['18-24', '25-34', '35-44'])

In [38]:
df['CHURCH_CAT'] = df['RELIGCAT'].apply(lambda x: 'No religion' if x == 'no religion' else None)
df.loc[df['CHATTND2'].isin(['once a month', '2 or 3 times a month', 'once a week', 'several times a week']), 'CHURCH_CAT'] = 'Regular'
df.loc[df['CHATTND2'].isin(['once a year', 'several times a year', 'less frequently', 'never']), 'CHURCH_CAT'] = 'Occasional or never'
df.loc[df['year'] < 2022, 'CHURCH_CAT'] = None

# Party dot plot

In [39]:
trav_cols = [
    'TRAVRES2',
    'TRAVCOLL',
    'TRAVREL',
]

ee_cols = [
    'EERES',
    'EECOLL',
    'EEREL',
]

muslim_cols = [
    'MUSRES2',
    'MUSCOLL2',
    'MUSREL2',
]

meg_cols = [
    'MEGRES',
    'MEGCOLL',
    'MEGREL',
]

In [40]:
migrants_asylum_cols_bin = ['SUPPORT_ASYLUM_SEEKERS', 'SUPPORT_REFUGEES', 'BELIEVE_REFUGEES_CONTRIB']

In [41]:
mix_rel_cols_bin = ['PREFER_MIXED_SCHOOL', 'PREFER_MIXED_NEIGHBOURHOOD', 'PREFER_MIXED_WORKPLACE']

In [42]:
bin_cols = trav_cols + ee_cols + muslim_cols + meg_cols + migrants_asylum_cols_bin + mix_rel_cols_bin

In [43]:
def crosstab(df, indep_cols, dep_col, pct=True):
    if type(indep_cols) is str:
        indep_cols = [indep_cols]
    ret = df.groupby([*indep_cols, dep_col], observed=True)['WTFACTOR'].sum().unstack().fillna(0)
    if pct is True:
        return ret.apply(lambda x: x/x.sum(), axis=1)
    else:
        return ret

In [44]:
def get_n(df, indep_cols, dep_col, pct=True):
    if type(indep_cols) is str:
        indep_cols = [indep_cols]
    return df.groupby([*indep_cols, dep_col], observed=True)['WTFACTOR'].sum().unstack().sum(axis='columns')

In [45]:
party_dot_plot = pd.concat([crosstab(df, 'POLITICAL_PARTY', col).T for col in bin_cols], keys=bin_cols).xs('yes', level=1)
party_dot_plot.to_csv('plot_data/party_dot_plot.csv')

In [46]:
party_dot_plot_N = pd.concat([get_n(df, 'POLITICAL_PARTY', col) for col in bin_cols], keys=bin_cols).reset_index().rename(columns={'level_0': 'question', 'POLITICAL_PARTY': 'split_1', 0: 'N'})

In [47]:
party_dot_plot_N = party_dot_plot_N[~(party_dot_plot_N['split_1'] == 'Other or none')]

# Chi-squared analysis

In [48]:
def get_cramers_v(chi2, n, r, c):
    denom = n * min(r - 1, c - 1)
    if denom == 0:
        return np.nan
    return np.sqrt(chi2 / denom)

In [49]:
def get_pvalue(df, indep_col, dep_col):
    table = crosstab(df, indep_col, dep_col, pct=False)
    chi2, p, dof, expected = chi2_contingency(table)
    n = table.sum().sum()
    r, c = table.shape
    cramers_v = get_cramers_v(chi2, n, r, c)
    return {'pvalue': p, 'cramers_v': cramers_v}

## Column groups

In [50]:
qs = pd.read_csv('nilt_data/question_text.csv').sort_values('category').drop_duplicates('question_code', keep='first').set_index('question_code')

In [51]:
q_text_dict = dict(qs['question_text'].drop_duplicates())

In [52]:
q_text_dict['PAYBILLSC']

'How well has your household kept up with bills and credit commitments in the past year?'

In [53]:
background_cols = list(qs[qs['category'] == 'Background'].index)

In [54]:
background_and_id_cols = background_cols + ['POLPART2', 'IRBRIT', 'UNINATID']

## Analysis

In [55]:
def top_n(df, id_col, n, columns=None):
    columns = set(df.columns) & set(columns) if columns else df.columns
    cols = [c for c in columns if df[c].isna().all() == False]
    pvals = pd.DataFrame([get_pvalue(df, id_col, c) for c in cols], index=cols)
    idx = list(set(qs.index) & set(pvals.index))
    pvals = pvals.loc[idx]
    ret = pvals.sort_values('cramers_v', ascending=False).head(n)
    ret['q_text'] = ret.index.map(q_text_dict)
    return ret

In [56]:
top_vars = top_n(df, 'MUSRES2', None, background_and_id_cols)
top_vars.head()

,pvalue,cramers_v,q_text
POLPART2,2.942235e-91,0.249016,Political party support
IRBRIT,3.539086e-59,0.198663,Do you see yourself as Irish or British?
UNINATID,1.342442e-59,0.196519,"Do you think of yourself as a unionist, a nati..."
PROTCATH,3.518051e-52,0.180459,Do you see yourself as part of the Protestant ...
HIGHQUAL2,3.068093e-45,0.176650,Highest educational qualification of respondent


In [57]:
def get_url(q_code):
    if q_code in ['POLPART2', 'IRBRIT', 'UNINATID']:
        return f'<a href="https://www.ark.ac.uk/nilt/2025/Political_Attitudes/{q_code}.html">{q_code}</a>'
    else:
        return f'<a href="https://www.ark.ac.uk/nilt/2025/Background/{q_code}.html">{q_code}</a>'

In [58]:
chi_squared_table = top_vars.reset_index().rename(columns={
    'index': 'Question Code',
    'pvalue': 'P-value',
    'cramers_v': 'Cramérs V statistic',
    'q_text': 'Question'
})

chi_squared_table['P-value'] = chi_squared_table['P-value'].apply(lambda x: f'{x:.1e}')
chi_squared_table['Question Code'] = chi_squared_table['Question Code'].apply(get_url)
chi_squared_table.to_csv('plot_data/chi_squared_table.csv', index=False)

# National Identity column chart

In [59]:
representative_questions = ['MUSRES2', 'SUPPORT_REFUGEES', 'PREFER_MIXED_NEIGHBOURHOOD']

In [60]:
a = crosstab(df, 'IRBRIT', 'MUSRES2')['yes'].rename('Would willingly accept a Muslim as a resident in their local area')

In [61]:
b = crosstab(df, 'IRBRIT', 'SUPPORT_REFUGEES')['yes'].rename('Agree that it is our duty to provide protection to refugees')

In [62]:
c = crosstab(df, 'IRBRIT', 'PREFER_MIXED_NEIGHBOURHOOD')['yes'].rename('Would prefer to live in a mixed neighbourhood')

In [63]:
national_identity_column_chart = pd.concat([a,b,c], axis='columns').drop(index=["Don't know", 'Other'])

In [64]:
national_identity_column_chart.to_csv('plot_data/national_identity_column_chart.csv')

In [65]:
national_identity_column_chart_N = (
    pd.concat([get_n(df, 'IRBRIT', col) for col in representative_questions], keys=representative_questions)
    .reset_index()
    .rename(columns={'level_0': 'question', 'IRBRIT': 'split_1', 0: 'N'})
)

national_identity_column_chart_N = national_identity_column_chart_N.loc[~national_identity_column_chart_N['split_1'].isin(['Other', "Don't know"])]

# Granular dot plots

In [66]:
def crosstab_irbrit(df, other_col, col_name_mapper):
    qcols = ['MUSRES2', 'SUPPORT_REFUGEES', 'PREFER_MIXED_NEIGHBOURHOOD']
    tab = pd.concat([(
        crosstab(df.replace({'yes': 1.0, 'no': 0.0}), ['IRBRIT', other_col], c)
        .unstack()[1.0]
        .drop(index=["Don't know", 'Other'])
        .rename(columns=col_name_mapper)
        * 100
    ) for c in qcols], keys=qcols).swaplevel()
    return tab.reset_index().rename(columns={'level_1': 'question'}).replace({
        'MUSRES2': 'Would willingly accept a Muslim as a resident in their local area',
        'SUPPORT_REFUGEES': 'Agree that it is our duty to provide protection to refugees',
        'PREFER_MIXED_NEIGHBOURHOOD': 'Would prefer to live in a mixed neighbourhood',
        'Irish not British': ':ie: only',
        'British not Irish': ':gb: only',
        'More Irish than British': 'More :ie:',
        'More British than Irish': 'More :gb:',
        'Equally Irish and British': 'Both equally'
    }).set_index(['IRBRIT', 'question'])

In [67]:
questions = [
    'Would willingly accept a Muslim as a resident in their local area',
    'Agree that it is our duty to provide protection to refugees',
    'Would prefer to live in a mixed neighbourhood'
]

order = [(n, q) for q in questions for n in [
    ':ie: only',
    'More :ie:',
    'Both equally',
    'More :gb:',
    ':gb: only'
]]

In [68]:
qcols = ['MUSRES2', 'SUPPORT_REFUGEES', 'PREFER_MIXED_NEIGHBOURHOOD']

## HAS_DEGREE

In [69]:
has_degree_dot_plot = crosstab_irbrit(df, 'HAS_DEGREE', {0.0: 'No degree', 1.0: 'Has degree'}).loc[order]

In [70]:
has_degree_dot_plot.to_csv('plot_data/has_degree_dot_plot.csv')

In [71]:
has_degree_dot_plot_N = pd.concat([get_n(df, ['IRBRIT', 'HAS_DEGREE'], q) for q in qcols], keys=qcols).reset_index().rename(columns={'level_0': 'question', 'IRBRIT': 'split_1', 'HAS_DEGREE': 'split_2', 0: 'N'})
has_degree_dot_plot_N['split_2'] = has_degree_dot_plot_N['split_2'].replace({False: 'No degree', True: 'Has degree'})
has_degree_dot_plot_N = has_degree_dot_plot_N.loc[~has_degree_dot_plot_N['split_1'].isin(['Other', "Don't know"])]

## INCOME

In [72]:
income_dot_plot = crosstab_irbrit(df, 'SELF_REPORTED_LOW_INCOME', {0.0: 'Medium or High Income', 1.0: 'Low Income'}).loc[order]

In [73]:
income_dot_plot.to_csv('plot_data/income_dot_plot.csv')

In [74]:
income_dot_plot_N = pd.concat([get_n(df, ['IRBRIT', 'SELF_REPORTED_LOW_INCOME'], q) for q in qcols], keys=qcols).reset_index().rename(columns={'level_0': 'question', 'IRBRIT': 'split_1', 'SELF_REPORTED_LOW_INCOME': 'split_2', 0: 'N'})
income_dot_plot_N['split_2'] = income_dot_plot_N['split_2'].replace({False: 'Medium or High Income', True: 'Low Income'})
income_dot_plot_N = income_dot_plot_N.loc[~income_dot_plot_N['split_1'].isin(['Other', "Don't know"])]

## LIVED OUT OF NI

In [75]:
outofni_dot_plot = crosstab_irbrit(df, 'OUTOFNI', {0.0: 'Always lived in NI', 1.0: 'Has lived outside of NI'}).loc[order]

In [76]:
outofni_dot_plot.to_csv('plot_data/outofni_dot_plot.csv')

In [77]:
outofni_dot_plot_N = pd.concat([get_n(df, ['IRBRIT', 'OUTOFNI'], q) for q in qcols], keys=qcols).reset_index().rename(columns={'level_0': 'question', 'IRBRIT': 'split_1', 'OUTOFNI': 'split_2', 0: 'N'})
outofni_dot_plot_N['split_2'] = outofni_dot_plot_N['split_2'].replace({'no': 'Always lived in NI', 'yes': 'Has lived outside of NI'})
outofni_dot_plot_N = outofni_dot_plot_N.loc[~outofni_dot_plot_N['split_1'].isin(['Other', "Don't know"])]

## AGE

In [78]:
age_dot_plot = crosstab_irbrit(df, 'UNDER_45', {0.0: '45 or over', 1.0: 'Under 45'}).loc[order]

In [79]:
age_dot_plot.to_csv('plot_data/age_dot_plot.csv')

In [80]:
age_dot_plot_N = pd.concat([get_n(df, ['IRBRIT', 'UNDER_45'], q) for q in qcols], keys=qcols).reset_index().rename(columns={'level_0': 'question', 'IRBRIT': 'split_1', 'UNDER_45': 'split_2', 0: 'N'})
age_dot_plot_N['split_2'] = age_dot_plot_N['split_2'].replace({False: '45 or over', True: 'Under 45'})
age_dot_plot_N = age_dot_plot_N.loc[~age_dot_plot_N['split_1'].isin(['Other', "Don't know"])]

## CHURCH ATTENDANCE

In [81]:
church_dot_plot = crosstab_irbrit(df, 'CHURCH_CAT', lambda x:x).loc[order]

In [82]:
church_dot_plot.to_csv('plot_data/church_dot_plot.csv')

In [83]:
church_dot_plot_N = pd.concat([get_n(df, ['IRBRIT', 'CHURCH_CAT'], q) for q in qcols], keys=qcols).reset_index().rename(columns={'level_0': 'question', 'IRBRIT': 'split_1', 'CHURCH_CAT': 'split_2', 0: 'N'})
church_dot_plot_N = church_dot_plot_N.loc[~church_dot_plot_N['split_1'].isin(['Other', "Don't know"])]

# N table

In [84]:
n_table = pd.concat([
    party_dot_plot_N,
    national_identity_column_chart_N,
    has_degree_dot_plot_N,
    income_dot_plot_N,
    outofni_dot_plot_N,
    age_dot_plot_N,
    church_dot_plot_N,
], keys=['Party Dot Plot', 'Column Chart', 'Education Dot Plot', 'Income Dot Plot', 'Lived Out Dot Plot', 'Age Dot Plot', 'Church Dot Plot'])

In [85]:
n_table = n_table.fillna('—')

In [87]:
n_table['Question code'] = n_table['question'].apply(lambda x: get_url(x) if len(x) < 10 else x)

In [88]:
n_table['Question'] = n_table['question'].apply(lambda x: q_text_dict.get(x, 'Derived variable - see question code'))

In [89]:
n_table.rename(columns={
    'split_1': 'First Split',
    'split_2': 'Second Split',
})[['Question code', 'First Split', 'Second Split', 'N']].reset_index(level=0).rename(columns={'level_0': 'Chart'}).to_csv('plot_data/n_table.csv', index=False)